# **Upload Data (Silver Layer Output)**

In [3]:
import pandas as pd
import numpy as np

df = pd.read_excel('cleaned_amazon_sales.xlsx')

df['Date'] = pd.to_datetime(df['Date'])

print("Shape:", df.shape)
df.head()

Shape: (13323, 35)


,Date,Title,Account Title,Market Place,SKU,FNSKU,ASIN,Parent ASIN,Is Parent,Brand,...,Impressions,Clicks,PPC Orders,PPC Sales,PPC Cost,PPC Conv,OOE,Net Profit,Net Margin,Net ROI
0,2020-10-01,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,3423,13,2,29.36,-13.06,0.1538,0,56.28,0.1852,0.3142
1,2020-10-02,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,2786,7,0,0.00,-7.33,0.0000,0,71.73,0.2034,0.3439
2,2020-10-03,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,2927,7,1,14.69,-6.15,0.1429,0,61.37,0.1386,0.2416
3,2020-10-04,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,3584,16,2,44.07,-16.51,0.1250,0,39.49,0.1573,0.2684
4,2020-10-05,High Absorption Magnesium Citrate Supplements ...,[ UK ] Aava,UK,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,...,4329,7,0,0.00,-4.47,0.0000,0,59.86,0.2116,0.3588


# **Gold Layer - Star Schema Design**

- **Dim Date**
- **Dim Product**
- **Dim Marketplace**
- **Dim Account**
- **Fact Sales**


## Dim Date

In [4]:
dim_date = pd.DataFrame({'Date': df['Date'].drop_duplicates().sort_values().reset_index(drop=True)})

dim_date['Date_Key'] = dim_date.index + 1

dim_date['Day'] = dim_date['Date'].dt.day
dim_date['Month'] = dim_date['Date'].dt.month
dim_date['Month_Name'] = dim_date['Date'].dt.month_name()
dim_date['Year'] = dim_date['Date'].dt.year
dim_date['Quarter'] = dim_date['Date'].dt.quarter
dim_date['Week_Of_Year'] = dim_date['Date'].dt.isocalendar().week
dim_date['Day_Name'] = dim_date['Date'].dt.day_name()
dim_date['Is_Weekend'] = dim_date['Day_Name'].isin(['Saturday', 'Sunday'])

dim_date = dim_date[['Date_Key', 'Date', 'Day', 'Month', 'Month_Name',
                      'Quarter', 'Year', 'Week_Of_Year', 'Day_Name', 'Is_Weekend']]

print("Dim Date Shape:", dim_date.shape)
dim_date.head()

Dim Date Shape: (209, 10)


,Date_Key,Date,Day,Month,Month_Name,Quarter,Year,Week_Of_Year,Day_Name,Is_Weekend
0,1,2020-10-01,1,10,October,4,2020,40,Thursday,False
1,2,2020-10-02,2,10,October,4,2020,40,Friday,False
2,3,2020-10-03,3,10,October,4,2020,40,Saturday,True
3,4,2020-10-04,4,10,October,4,2020,40,Sunday,True
4,5,2020-10-05,5,10,October,4,2020,41,Monday,False


## Dim Product

In [5]:
dim_product = df[['SKU', 'FNSKU', 'ASIN', 'Parent ASIN', 'Is Parent', 'Brand', 'Title']].drop_duplicates().reset_index(drop=True)

dim_product['Product_Key'] = dim_product.index + 1

dim_product = dim_product[['Product_Key', 'SKU', 'FNSKU', 'ASIN', 'Parent ASIN',
                            'Is Parent', 'Brand', 'Title']]

print("Dim Product Shape:", dim_product.shape)
dim_product.head()

Dim Product Shape: (102, 8)


,Product_Key,SKU,FNSKU,ASIN,Parent ASIN,Is Parent,Brand,Title
0,1,UK_Aava_Magnesium_400mg_120_05,X000JJIQ79,B01GU9QLZC,No Parent,0,Aavalabs,High Absorption Magnesium Citrate Supplements ...
1,2,UK_Aava_Omega_2000mg_120_05,X000JFSY57,B01GU9P0M2,No Parent,0,Aavalabs,Omega 3 2000mg Fish Oil Capsules - High Streng...
2,3,UK_Aava_Turmeric_1400mg_120_07,X000J6KEPJ,B01GU8L68A,No Parent,0,Aavalabs,Turmeric Curcumin Capsules with Black Pepper [...
3,4,FR_Aava_Turmeric_1400mg_180_01,X000UL0561,B07CXYM3H5,No Parent,0,Aavalabs,Gélules biodisponibles du complexe de curcumin...
4,5,FR_Aava_Magnesium_400mg_120_01,X000UL0OUX,B07CZ1P4BM,No Parent,0,Aavalabs,"Citrate de Magnésium - 1496 mg dont 448,8 mg d..."


## Dim Marketplace

In [6]:
dim_marketplace = pd.DataFrame({'Market Place': df['Market Place'].drop_duplicates().reset_index(drop=True)})

dim_marketplace['Marketplace_Key'] = dim_marketplace.index + 1

dim_marketplace = dim_marketplace[['Marketplace_Key', 'Market Place']]

print("Dim Marketplace Shape:", dim_marketplace.shape)
dim_marketplace.head()

Dim Marketplace Shape: (5, 2)


,Marketplace_Key,Market Place
0,1,UK
1,2,FR
2,3,DE
3,4,IT
4,5,ES


## Dim Account

In [7]:
dim_account = pd.DataFrame({'Account Title': df['Account Title'].drop_duplicates().reset_index(drop=True)})

dim_account['Account_Key'] = dim_account.index + 1

dim_account = dim_account[['Account_Key', 'Account Title']]

print("Dim Account Shape:", dim_account.shape)
dim_account.head()

Dim Account Shape: (5, 2)


,Account_Key,Account Title
0,1,[ UK ] Aava
1,2,[ FR ] Aava
2,3,[ DE ] Aava
3,4,[ IT ] Aava
4,5,[ ES ] Aava


## Fact Sales
Join the original Silver data back to each dimension to pull in the surrogate keys, then keep only the keys plus the measures.

In [8]:
fact_sales = df.merge(dim_date[['Date_Key', 'Date']], on='Date', how='left')

fact_sales = fact_sales.merge(
    dim_product[['Product_Key', 'SKU', 'FNSKU', 'ASIN', 'Parent ASIN', 'Is Parent', 'Brand', 'Title']],
    on=['SKU', 'FNSKU', 'ASIN', 'Parent ASIN', 'Is Parent', 'Brand', 'Title'],
    how='left'
)

fact_sales = fact_sales.merge(dim_marketplace, on='Market Place', how='left')

fact_sales = fact_sales.merge(dim_account, on='Account Title', how='left')

fact_sales['Sales_Key'] = fact_sales.index + 1

measure_cols = [
    'Taxes', 'Orders', 'Units', 'Refunded', 'Refund %', 'Unit Session %',
    'Promo Units', 'Organic Units', 'Per Unit Revenue', 'Revenue', 'COGS',
    'FBA Fees', 'Promo Amount', 'Sessions', 'Page Views', 'Impressions',
    'Clicks', 'PPC Orders', 'PPC Sales', 'PPC Cost', 'PPC Conv', 'OOE',
    'Net Profit', 'Net Margin', 'Net ROI'
]

fact_sales = fact_sales[
    ['Sales_Key', 'Date_Key', 'Product_Key', 'Marketplace_Key', 'Account_Key'] + measure_cols
]

print("Fact Sales Shape:", fact_sales.shape)
fact_sales.head()

Fact Sales Shape: (13323, 30)


,Sales_Key,Date_Key,Product_Key,Marketplace_Key,Account_Key,Taxes,Orders,Units,Refunded,Refund %,...,Impressions,Clicks,PPC Orders,PPC Sales,PPC Cost,PPC Conv,OOE,Net Profit,Net Margin,Net ROI
0,1,1,1,1,1,-50.34,17,17,0,0.0,...,3423,13,2,29.36,-13.06,0.1538,0,56.28,0.1852,0.3142
1,2,2,1,1,1,-58.89,20,20,0,0.0,...,2786,7,0,0.00,-7.33,0.0000,0,71.73,0.2034,0.3439
2,3,3,1,1,1,-72.41,18,24,0,0.0,...,2927,7,1,14.69,-6.15,0.1429,0,61.37,0.1386,0.2416
3,4,4,1,1,1,-41.43,13,14,0,0.0,...,3584,16,2,44.07,-16.51,0.1250,0,39.49,0.1573,0.2684
4,5,5,1,1,1,-47.04,16,16,0,0.0,...,4329,7,0,0.00,-4.47,0.0000,0,59.86,0.2116,0.3588


## Validation Checks

In [9]:
print("Original Silver rows:", df.shape[0])
print("Fact Sales rows:", fact_sales.shape[0])

print("\nNull keys check:")
print(fact_sales[['Date_Key', 'Product_Key', 'Marketplace_Key', 'Account_Key']].isnull().sum())

print("\nTotal Revenue check (Silver vs Fact):", df['Revenue'].sum(), "vs", fact_sales['Revenue'].sum())

Original Silver rows: 13323
Fact Sales rows: 13323

Null keys check:
Date_Key           0
Product_Key        0
Marketplace_Key    0
Account_Key        0
dtype: int64

Total Revenue check (Silver vs Fact): 3628133.7699999996 vs 3628133.7699999996


# **Create Aggregated Gold Marts (Optional Business-Ready Views)**

## Monthly Sales Summary

In [10]:
fact_with_date = fact_sales.merge(dim_date, on='Date_Key', how='left')

monthly_sales_summary = fact_with_date.groupby(['Year', 'Month', 'Month_Name']).agg(
    Total_Orders=('Orders', 'sum'),
    Total_Units=('Units', 'sum'),
    Total_Revenue=('Revenue', 'sum'),
    Total_Net_Profit=('Net Profit', 'sum'),
    Avg_Net_Margin=('Net Margin', 'mean')
).reset_index().sort_values(['Year', 'Month'])

monthly_sales_summary.head()

,Year,Month,Month_Name,Total_Orders,Total_Units,Total_Revenue,Total_Net_Profit,Avg_Net_Margin
0,2020,10,October,22566,26582,556418.12,119790.89,1.656896e-01
1,2020,11,November,23112,27768,567418.18,129484.34,1.699518e+12
2,2020,12,December,16944,19982,421130.36,99552.85,1.583924e-01
3,2021,1,January,23987,27944,561266.13,118611.42,1.630335e-01
4,2021,2,February,21542,25015,496282.89,105413.24,1.601643e-01


## Brand Performance Summary

In [11]:
fact_with_product = fact_sales.merge(dim_product, on='Product_Key', how='left')

brand_performance = fact_with_product.groupby('Brand').agg(
    Total_Orders=('Orders', 'sum'),
    Total_Units=('Units', 'sum'),
    Total_Revenue=('Revenue', 'sum'),
    Total_Net_Profit=('Net Profit', 'sum'),
    Avg_Net_ROI=('Net ROI', 'mean')
).reset_index().sort_values('Total_Revenue', ascending=False)

brand_performance.head()

,Brand,Total_Orders,Total_Units,Total_Revenue,Total_Net_Profit,Avg_Net_ROI
0,Aavalabs,151164,177177,3628133.77,815690.23,0.294288


## Marketplace Performance Summary

In [12]:
fact_with_market = fact_sales.merge(dim_marketplace, on='Marketplace_Key', how='left')

marketplace_performance = fact_with_market.groupby('Market Place').agg(
    Total_Orders=('Orders', 'sum'),
    Total_Units=('Units', 'sum'),
    Total_Revenue=('Revenue', 'sum'),
    Total_Net_Profit=('Net Profit', 'sum'),
    Avg_Net_Margin=('Net Margin', 'mean')
).reset_index().sort_values('Total_Revenue', ascending=False)

marketplace_performance.head()

,Market Place,Total_Orders,Total_Units,Total_Revenue,Total_Net_Profit,Avg_Net_Margin
0,DE,45988,55390,1151257.96,333395.70,2.439859e-01
1,ES,45381,52445,1111192.69,260852.48,2.045839e-01
3,IT,23919,28012,589495.20,105717.83,1.117406e-01
4,UK,27396,30880,566358.12,78125.58,4.542503e+12
2,FR,8480,10450,209829.80,37598.64,-3.436943e+12


# **Download Gold Layer Tables**

In [13]:
with pd.ExcelWriter('gold_layer_amazon_sales.xlsx', engine='openpyxl') as writer:

    dim_date.to_excel(writer, index=False, sheet_name='Dim_Date')
    dim_product.to_excel(writer, index=False, sheet_name='Dim_Product')
    dim_marketplace.to_excel(writer, index=False, sheet_name='Dim_Marketplace')
    dim_account.to_excel(writer, index=False, sheet_name='Dim_Account')
    fact_sales.to_excel(writer, index=False, sheet_name='Fact_Sales')
    monthly_sales_summary.to_excel(writer, index=False, sheet_name='Mart_Monthly_Sales')
    brand_performance.to_excel(writer, index=False, sheet_name='Mart_Brand_Performance')
    marketplace_performance.to_excel(writer, index=False, sheet_name='Mart_Marketplace_Performance')

print("Gold layer Excel file saved successfully!")

Gold layer Excel file saved successfully!


In [14]:
from google.colab import files

files.download('gold_layer_amazon_sales.xlsx')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>